# Module 1: Structured Summary Builder

This notebook implements the structured-summary stage of the final project. It converts raw cross-batch RNA-seq classification result files into clean, validated, and LLM-ready summary tables for later pattern detection and LLM-assisted interpretation.

**Inputs:**  
- Raw cross-batch robustness result files from the experiment output folder  
- File names containing experimental metadata such as classifier, scenario, batch-balance setting, normalization method, and split ratio

**Processing steps:**  
1. Load raw result files from the experiment output directory.  
2. Validate file names and required data columns.  
3. Extract experimental metadata from each file name.  
4. Reshape wide-format result files into tidy long-format tables.  
5. Aggregate error values by classifier, scenario, normalization method, and split setting.  
6. Save both performance summaries and metadata summaries for downstream modules.

**Outputs:**  
All generated CSV and JSON files are saved under:

`module1_outputs/`

Key output files include:

- `module1_tidy_results.csv`
- `module1_performance_summary.csv`
- `module1_metadata_summary.json`

This module supports the final project repository requirements by making raw ML experiment outputs clean, documented, reproducible, and easy to use in later analysis modules.


## 1. Environment setup

This section loads dependencies in a reproducible way. The helper keeps local installations isolated, which is useful for GitHub review, demo execution, and repeated runs on a different machine.


In [ ]:
# ### Environment and dependency setup
# This cell imports core libraries and defines a small dependency helper so the notebook can run reproducibly in local or Colab-style environments.

from pathlib import Path
import importlib
import importlib.util
import json
import re
import subprocess
import sys

LOCAL_DEPS = Path.cwd() / ".pydeps"
if LOCAL_DEPS.exists():
    sys.path.insert(0, str(LOCAL_DEPS))


# ### Function: ensure_package
# Install or import one required package, then verify optional attributes so later cells fail early with useful messages.
def ensure_package(
    import_name: str,
    pip_name: str | None = None,
    required_attrs: tuple[str, ...] = (),
):
    """Install and import a Python package needed by the notebook.

    Inputs:
        import_name: Module name used in an import statement.
        pip_name: Optional package name used by pip when it differs from import_name.
        required_attrs: Attributes that must exist on the imported module.

    Output:
        Imported Python module object.
    """
    package_name = pip_name or import_name
    needs_install = importlib.util.find_spec(import_name) is None

    if not needs_install:
        module = importlib.import_module(import_name)
        needs_install = any(not hasattr(module, attr) for attr in required_attrs)
    else:
        module = None

    if needs_install:
        print(f"Installing missing package locally: {package_name}")
        LOCAL_DEPS.mkdir(exist_ok=True)
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                "--target",
                str(LOCAL_DEPS),
                package_name,
            ]
        )
        sys.path.insert(0, str(LOCAL_DEPS))
        importlib.invalidate_caches()
        sys.modules.pop(import_name, None)
        module = importlib.import_module(import_name)

    missing_attrs = [attr for attr in required_attrs if not hasattr(module, attr)]
    if missing_attrs:
        raise ImportError(
            f"Package {import_name!r} imported but is missing required attributes: {missing_attrs}"
        )

    return importlib.import_module(import_name)


pd = ensure_package("pandas", required_attrs=("read_csv",))


## 2. Expected experiment schema

This section documents the fixed experimental design used by Module 1: expected split values, supported classifiers, normalization names, file-name conventions, and final output schemas.


In [ ]:
# ### Module 1 constants and schema definitions
# These constants define the expected experimental design, supported classifiers, file-name parser, and output table schemas.

EXPECTED_SPLITS = [50, 70, 80, 90, 100]
EXPECTED_NORMALIZATIONS = ["non", "qn", "mn", "vsn"]
SUPPORTED_CLASSIFIERS = ("knn", "lasso", "pam", "rf", "svm", "xgb")

RESULT_FILE_PATTERN = re.compile(
    r"^(?P<scenario>jama_scenario\d+(?:_\d+)*)_"
    r"(?P<classifier>knn|lasso|pam|rf|svm|xgb)_batch_balance_"
    r"(?P<batch_balance>TRUE|FALSE)_result\.csv$"
)
RESULT_COLUMN_PATTERN = re.compile(r"^(?P<normalization>[A-Za-z0-9]+)\.(?P<repeat_id>\d+)$")

TIDY_COLUMNS = [
    "scenario",
    "classifier",
    "batch_balance",
    "normalization",
    "repeat_id",
    "split",
    "error",
]
SUMMARY_COLUMNS = [
    "scenario",
    "classifier",
    "batch_balance",
    "normalization",
    "split",
    "error_mean",
    "error_sd",
    "error_min",
    "error_max",
    "n_runs",
]


## 3. Function map

The main functions are organized as follows:

| Function | Role in the pipeline |
|---|---|
| `read_result_file` | Reads one raw result file and validates the split/error columns. |
| `parse_file_metadata` | Extracts scenario, classifier, and batch-balance information from the file name. |
| `convert_wide_to_tidy` | Converts raw wide-format files into one-row-per-error-observation tidy format. |
| `summarize_performance` | Aggregates repeated runs into mean/sd/min/max error summaries. |
| `build_metadata_summary` | Creates a JSON audit trail for reproducibility. |
| `build_structured_summary` | Runs the full Module 1 workflow and saves all deliverables. |

This design makes the notebook easier to grade against repository quality, documentation, reproducibility, and successful implementation criteria.


In [ ]:
# ### Core Module 1 functions
# This cell implements the complete transformation from raw ML result files to tidy results, aggregated summaries, and metadata.
# The functions are intentionally separated into small validation, parsing, reshaping, aggregation, and final-check steps for repository readability and reproducibility.

# ### Function: read_result_file
# Load one raw result CSV/TSV file and perform basic validation before reshaping.
def read_result_file(file_path: str | Path):
    """Read and validate one wide experiment result file.

    Inputs:
        file_path: Path to one result file. Files may be comma- or tab-separated.

    Output:
        DataFrame with integer `split` column and numeric error columns.
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Result file does not exist: {path}")

    df = pd.read_csv(path)
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep="	")

    if df.empty:
        raise ValueError(f"Result file is empty: {path}")

    df = _normalize_split_column(df, path)
    _validate_split_values(df, path)
    df = _convert_error_columns_to_numeric(df, path)
    return df


# ### Helper: _normalize_split_column
# Standardize the split column so downstream grouping uses numeric split values consistently.
def _normalize_split_column(df, path: Path):
    """Ensure `split` is a normal integer column.

    Inputs:
        df: Raw DataFrame read from one result file.
        path: Source file path used for clear error messages.

    Output:
        DataFrame with a normalized integer `split` column.
    """
    df = df.copy()
    first_column = str(df.columns[0]).strip()

    if "split" in df.columns:
        pass
    elif first_column == "" or first_column.startswith("Unnamed:") or _looks_like_split_column(df.iloc[:, 0]):
        df = df.rename(columns={df.columns[0]: "split"})
    else:
        raise ValueError(
            f"{path.name}: could not identify the split column. "
            "Expected a 'split' column or an unnamed/index-like first column."
        )

    converted_split = pd.to_numeric(df["split"], errors="coerce")
    if converted_split.isna().any():
        bad_values = df.loc[converted_split.isna(), "split"].astype(str).tolist()
        raise ValueError(f"{path.name}: split column contains non-numeric values: {bad_values}")

    if not (converted_split % 1 == 0).all():
        bad_values = df.loc[(converted_split % 1) != 0, "split"].astype(str).tolist()
        raise ValueError(f"{path.name}: split column contains non-integer values: {bad_values}")

    df["split"] = converted_split.astype(int)
    return df


# ### Helper: _looks_like_split_column
# Detect whether a column appears to encode split percentages such as 50, 70, 80, 90, or 100.
def _looks_like_split_column(series) -> bool:
    """Check whether a series exactly matches expected split values.

    Inputs:
        series: Candidate first column from a result table.

    Output:
        True when the series is numeric and equals EXPECTED_SPLITS.
    """
    converted = pd.to_numeric(series, errors="coerce")
    if converted.isna().any():
        return False
    return converted.astype(int).tolist() == EXPECTED_SPLITS


# ### Helper: _validate_split_values
# Check that observed split values match the expected cross-batch split design.
def _validate_split_values(df, path: Path) -> None:
    """Validate that result rows use the expected split values.

    Inputs:
        df: DataFrame containing a `split` column.
        path: Source file path used for clear error messages.

    Output:
        None. Raises ValueError if splits differ from EXPECTED_SPLITS.
    """
    actual_splits = df["split"].tolist()
    if actual_splits != EXPECTED_SPLITS:
        raise ValueError(
            f"{path.name}: split values must be exactly {EXPECTED_SPLITS}; "
            f"found {actual_splits}."
        )


# ### Helper: _convert_error_columns_to_numeric
# Coerce normalization/repeat error columns to numeric values and raise clear errors for malformed data.
def _convert_error_columns_to_numeric(df, path: Path):
    """Convert all non-split result columns to numeric errors.

    Inputs:
        df: Wide result DataFrame with a `split` column.
        path: Source file path used for clear error messages.

    Output:
        DataFrame with numeric error columns.
    """
    df = df.copy()
    error_columns = [column for column in df.columns if column != "split"]
    failed_columns = []

    for column in error_columns:
        converted = pd.to_numeric(df[column], errors="coerce")
        if converted.isna().any():
            failed_columns.append(str(column))
        df[column] = converted

    if failed_columns:
        raise ValueError(
            f"{path.name}: these error columns could not be converted to numeric: "
            f"{', '.join(failed_columns)}"
        )

    return df


# ### Function: parse_file_metadata
# Extract scenario, classifier, and batch-balance settings from the standardized result file name.
def parse_file_metadata(file_path: str | Path) -> dict[str, object]:
    """Parse scenario metadata from a result filename.

    Inputs:
        file_path: Result filename or path matching the Module 1 naming convention.

    Output:
        Dictionary with `scenario`, `classifier`, and boolean `batch_balance`.
    """
    path = Path(file_path)
    match = RESULT_FILE_PATTERN.match(path.name)
    if not match:
        supported = ", ".join(SUPPORTED_CLASSIFIERS)
        raise ValueError(
            f"{path.name}: filename does not match the expected format "
            "'jama_scenario1_3_<classifier>_batch_balance_TRUE_result.csv' "
            f"with classifier in [{supported}] and batch_balance TRUE or FALSE."
        )

    metadata = match.groupdict()
    metadata["batch_balance"] = metadata["batch_balance"] == "TRUE"
    return metadata


# ### Function: convert_wide_to_tidy
# Convert one wide result table into long format with one row per scenario-classifier-normalization-split-repeat error value.
def convert_wide_to_tidy(df, metadata: dict[str, object]):
    """Convert a wide result table into long-format tidy rows.

    Inputs:
        df: Validated wide DataFrame from read_result_file().
        metadata: Parsed file metadata from parse_file_metadata().

    Output:
        Tidy DataFrame with columns defined by TIDY_COLUMNS.
    """
    if "split" not in df.columns:
        raise ValueError("Input DataFrame must contain a 'split' column.")

    actual_splits = sorted(int(value) for value in df["split"].unique().tolist())
    if actual_splits != EXPECTED_SPLITS:
        raise ValueError(f"split values must be exactly {EXPECTED_SPLITS}; found {actual_splits}.")

    long_df = df.melt(id_vars="split", var_name="normalization_repeat", value_name="error")
    parsed_columns = long_df["normalization_repeat"].str.extract(RESULT_COLUMN_PATTERN)

    invalid_columns = sorted(
        long_df.loc[parsed_columns.isna().any(axis=1), "normalization_repeat"].unique().tolist()
    )
    if invalid_columns:
        raise ValueError(
            "Invalid result column names. Expected '<normalization>.<repeat_id>'; "
            f"found {invalid_columns}."
        )

    long_df["normalization"] = parsed_columns["normalization"]
    invalid_normalizations = sorted(
        set(long_df["normalization"].unique().tolist()) - set(EXPECTED_NORMALIZATIONS)
    )
    if invalid_normalizations:
        raise ValueError(
            f"normalization must be one of {EXPECTED_NORMALIZATIONS}; "
            f"found {invalid_normalizations}."
        )

    long_df["repeat_id"] = pd.to_numeric(parsed_columns["repeat_id"], errors="coerce")
    if long_df["repeat_id"].isna().any() or not (long_df["repeat_id"] % 1 == 0).all():
        raise ValueError("repeat_id must be integer for every result column.")
    long_df["repeat_id"] = long_df["repeat_id"].astype(int)

    long_df["error"] = pd.to_numeric(long_df["error"], errors="coerce")
    if long_df["error"].isna().any():
        raise ValueError("error must be numeric for every tidy row.")

    long_df["scenario"] = metadata["scenario"]
    long_df["classifier"] = metadata["classifier"]
    long_df["batch_balance"] = metadata["batch_balance"]

    return long_df[TIDY_COLUMNS].sort_values(
        ["scenario", "classifier", "split", "repeat_id", "normalization"]
    ).reset_index(drop=True)


# ### Function: summarize_performance
# Aggregate tidy error values into mean, standard deviation, min, max, and run counts for each configuration.
def summarize_performance(tidy_df):
    """Summarize tidy classification errors by classifier, normalization, and split.

    Inputs:
        tidy_df: Full tidy DataFrame with columns defined by TIDY_COLUMNS.

    Output:
        Performance summary DataFrame with columns defined by SUMMARY_COLUMNS.
    """
    missing_columns = [column for column in TIDY_COLUMNS if column not in tidy_df.columns]
    if missing_columns:
        raise ValueError(f"tidy_df is missing required columns: {missing_columns}")

    tidy_df = tidy_df.copy()
    tidy_df["error"] = pd.to_numeric(tidy_df["error"], errors="coerce")
    if tidy_df["error"].isna().any():
        raise ValueError("error must be numeric before performance summarization.")

    group_columns = ["scenario", "classifier", "batch_balance", "normalization", "split"]
    summary = (
        tidy_df.groupby(group_columns, as_index=False)
        .agg(
            error_mean=("error", "mean"),
            error_sd=("error", "std"),
            error_min=("error", "min"),
            error_max=("error", "max"),
            n_runs=("repeat_id", "nunique"),
        )
        .sort_values(group_columns)
        .reset_index(drop=True)
    )

    unique_run_counts = sorted(int(value) for value in summary["n_runs"].unique().tolist())
    if len(unique_run_counts) != 1:
        print(f"Warning: n_runs is not consistent across groups: {unique_run_counts}")

    return summary[SUMMARY_COLUMNS]


# ### Helper: _ordered_unique
# Return unique values in a stable sorted order for readable metadata output.
def _ordered_unique(values, preferred_order: list[str] | None = None) -> list:
    """Return unique values in a stable preferred order.

    Inputs:
        values: Iterable of observed values.
        preferred_order: Optional order to apply before sorted extras.

    Output:
        List of unique values.
    """
    unique_values = list(dict.fromkeys(values))
    if preferred_order is None:
        return sorted(unique_values)
    preferred_values = [value for value in preferred_order if value in unique_values]
    extra_values = sorted(value for value in unique_values if value not in preferred_order)
    return preferred_values + extra_values


# ### Function: build_metadata_summary
# Create a compact JSON summary describing files, row counts, classifiers, normalizations, and split values.
def build_metadata_summary(result_files: list[Path], tidy_results, performance_summary) -> dict[str, object]:
    """Build metadata describing Module 1 output tables.

    Inputs:
        result_files: Files processed by build_structured_summary().
        tidy_results: Full tidy results DataFrame.
        performance_summary: Full performance summary DataFrame.

    Output:
        JSON-serializable metadata dictionary.
    """
    return {
        "number_of_files": len(result_files),
        "scenarios": sorted(tidy_results["scenario"].unique().tolist()),
        "classifiers": sorted(tidy_results["classifier"].unique().tolist()),
        "normalizations": _ordered_unique(
            tidy_results["normalization"].unique().tolist(), EXPECTED_NORMALIZATIONS
        ),
        "split_values": sorted(int(value) for value in tidy_results["split"].unique().tolist()),
        "batch_balance_values": sorted(tidy_results["batch_balance"].unique().tolist()),
        "total_tidy_rows": int(len(tidy_results)),
        "total_summary_rows": int(len(performance_summary)),
    }


# ### Function: build_structured_summary
# Run the full Module 1 pipeline: load raw files, reshape, aggregate, validate, and save CSV/JSON outputs.
def build_structured_summary(input_dir: str | Path, output_dir: str | Path | None = None):
    """Build and save all Module 1 structured summary outputs.

    Inputs:
        input_dir: Folder containing JAMA scenario result files.
        output_dir: Destination folder for Module 1 outputs. Defaults to input_dir.

    Output:
        Tuple of tidy results DataFrame, performance summary DataFrame, and metadata dict.
    """
    input_path = Path(input_dir)
    if not input_path.exists() or not input_path.is_dir():
        raise NotADirectoryError(f"Input folder does not exist or is not a directory: {input_path}")

    output_path = Path(output_dir) if output_dir else input_path
    output_path.mkdir(parents=True, exist_ok=True)

    result_files = sorted(input_path.glob("jama_scenario1_3_*_batch_balance_TRUE_result.csv"))
    if not result_files:
        raise ValueError(
            "No result files matching "
            "'jama_scenario1_3_*_batch_balance_TRUE_result.csv' "
            f"were found in {input_path}"
        )

    tidy_frames = []
    for file_path in result_files:
        metadata = parse_file_metadata(file_path)
        raw_df = read_result_file(file_path)
        tidy_frames.append(convert_wide_to_tidy(raw_df, metadata))

    tidy_results = pd.concat(tidy_frames, ignore_index=True)
    performance_summary = summarize_performance(tidy_results)

    _validate_final_outputs(tidy_results, performance_summary)

    tidy_output = output_path / "module1_tidy_results.csv"
    summary_output = output_path / "module1_performance_summary.csv"
    metadata_output = output_path / "module1_metadata_summary.json"

    tidy_results.to_csv(tidy_output, index=False)
    performance_summary.to_csv(summary_output, index=False)

    metadata = build_metadata_summary(result_files, tidy_results, performance_summary)
    metadata_output.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    return tidy_results, performance_summary, metadata


# ### Helper: _validate_final_outputs
# Confirm that generated Module 1 outputs contain the expected schemas and nonempty results.
def _validate_final_outputs(tidy_results, performance_summary) -> None:
    """Validate final Module 1 table schemas and expected dataset dimensions.

    Inputs:
        tidy_results: Full tidy results DataFrame.
        performance_summary: Full performance summary DataFrame.

    Output:
        None. Raises ValueError when validation fails.
    """
    if tidy_results.columns.tolist() != TIDY_COLUMNS:
        raise ValueError(f"Unexpected tidy columns: {tidy_results.columns.tolist()}")
    if performance_summary.columns.tolist() != SUMMARY_COLUMNS:
        raise ValueError(f"Unexpected summary columns: {performance_summary.columns.tolist()}")

    split_values = sorted(int(value) for value in tidy_results["split"].unique().tolist())
    if split_values != EXPECTED_SPLITS:
        raise ValueError(f"Expected split values {EXPECTED_SPLITS}; found {split_values}.")

    normalizations = _ordered_unique(tidy_results["normalization"].unique().tolist(), EXPECTED_NORMALIZATIONS)
    if normalizations != EXPECTED_NORMALIZATIONS:
        raise ValueError(f"Expected normalizations {EXPECTED_NORMALIZATIONS}; found {normalizations}.")

    if len(tidy_results) != 360:
        raise ValueError(f"Expected 360 tidy rows for current dataset; found {len(tidy_results)}.")
    if len(performance_summary) != 120:
        raise ValueError(
            f"Expected 120 performance summary rows for current dataset; found {len(performance_summary)}."
        )


## 4. Run the structured-summary pipeline

The following cell is the only execution step required for Module 1. It reads `./result`, writes `./module1_outputs`, and stores the returned objects for validation and demonstration.


In [ ]:
# ### Execute Module 1 pipeline
# Input: raw experiment result files in ./result. Output: tidy CSV, performance summary CSV, and metadata JSON in ./module1_outputs.

input_dir = "./result"
output_dir = "./module1_outputs"

tidy_results, performance_summary, metadata = build_structured_summary(input_dir, output_dir)


## 5. Validate outputs

This section prints a compact validation summary and uses assertions to make the expected output dimensions explicit. These checks help demonstrate reproducibility and make failures easier to diagnose.


In [ ]:
# ### Validation summary and reproducibility assertions
# These printed checks make the expected row counts, classifiers, normalizations, and split values explicit for grading and debugging.

print("Module 1 validation summary")
print(f"Number of files: {metadata['number_of_files']}")
print(f"Tidy rows: {metadata['total_tidy_rows']}")
print(f"Summary rows: {metadata['total_summary_rows']}")
print(f"Classifiers: {', '.join(metadata['classifiers'])}")
print(f"Normalizations: {', '.join(metadata['normalizations'])}")
print(f"Splits: {', '.join(str(value) for value in metadata['split_values'])}")

assert metadata["total_tidy_rows"] == 360, metadata["total_tidy_rows"]
assert metadata["total_summary_rows"] == 120, metadata["total_summary_rows"]
assert tidy_results.columns.tolist() == TIDY_COLUMNS, tidy_results.columns.tolist()
assert performance_summary.columns.tolist() == SUMMARY_COLUMNS, performance_summary.columns.tolist()
print("PASS: Module 1 output schemas and row counts are valid.")


## 6. Usage example and demonstration

The final cells provide a small KNN-focused example. This is not a separate experiment; it is a readable demonstration that the output tables can be inspected and used by downstream modules.


In [ ]:
# ### Example output checker
# This function prints a focused KNN example to demonstrate that the reshaped and summarized outputs are interpretable.

# ### Function: print_knn_example_outputs
# Print representative KNN rows and simple PASS/FAIL checks as a usage demonstration.
def print_knn_example_outputs(tidy_df, summary_df) -> None:
    """Print a KNN-focused sanity check for Module 1 outputs.

    Inputs:
        tidy_df: Full Module 1 tidy results DataFrame.
        summary_df: Full Module 1 performance summary DataFrame.

    Output:
        None. Prints example rows and PASS/FAIL validation messages.
    """
    try:
        knn_tidy = tidy_df[tidy_df["classifier"] == "knn"].copy()
        if knn_tidy.empty:
            print("FAIL: No KNN rows found in tidy_df.")
            raise ValueError("No KNN rows found in tidy_df.")

        print("First 12 rows of KNN tidy table:")
        print(knn_tidy.head(12).to_string(index=False))
        print()

        knn_summary = summary_df[summary_df["classifier"] == "knn"].copy()
        if knn_summary.empty:
            print("FAIL: No KNN rows found in summary_df.")
            raise ValueError("No KNN rows found in summary_df.")

        normalization_order = {value: index for index, value in enumerate(EXPECTED_NORMALIZATIONS)}
        knn_summary["_normalization_order"] = knn_summary["normalization"].map(normalization_order)
        knn_summary = (
            knn_summary.sort_values(["_normalization_order", "split"])
            .drop(columns="_normalization_order")
            .reset_index(drop=True)
        )

        print("KNN performance summary sorted by normalization and split:")
        print(knn_summary.to_string(index=False))
        print()

        split_values = sorted(int(value) for value in knn_tidy["split"].unique().tolist())
        if split_values == EXPECTED_SPLITS:
            print(f"PASS: KNN split values are exactly {EXPECTED_SPLITS}.")
        else:
            print(f"FAIL: KNN split values expected {EXPECTED_SPLITS}, found {split_values}.")
            raise ValueError(f"KNN split values expected {EXPECTED_SPLITS}, found {split_values}.")

        normalizations = _ordered_unique(knn_tidy["normalization"].unique().tolist(), EXPECTED_NORMALIZATIONS)
        if normalizations == EXPECTED_NORMALIZATIONS:
            print(f"PASS: KNN normalizations are exactly {EXPECTED_NORMALIZATIONS}.")
        else:
            print(
                f"FAIL: KNN normalizations expected {EXPECTED_NORMALIZATIONS}, found {normalizations}."
            )
            raise ValueError(
                f"KNN normalizations expected {EXPECTED_NORMALIZATIONS}, found {normalizations}."
            )

    except Exception:
        print("FAIL: KNN sanity check did not complete successfully.")
        raise


In [ ]:
# ### Demonstration call
# Run the KNN-focused example output check after the full summary pipeline has completed.

print_knn_example_outputs(tidy_results, performance_summary)
